# P5-4 · GĐ 4 · Latent diffusion và Stable Diffusion

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q diffusers

## Lý thuyết

In [ ]:
# Cần GPU và mạng: tải Stable Diffusion 1.5 (khoảng 2 GB ở fp16), chạy trên Colab T4
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
dev = "cuda"
pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5",
                                               torch_dtype=torch.float16).to(dev)
tok, te, unet, vae = pipe.tokenizer, pipe.text_encoder, pipe.unet, pipe.vae
sched = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

@torch.no_grad()
def encode(texts):
    ids = tok(texts, padding="max_length", max_length=tok.model_max_length,
              truncation=True, return_tensors="pt").input_ids.to(dev)
    return te(ids)[0]                                            # (B, 77, 768)

@torch.no_grad()
def generate(prompt, negative="", steps=30, w=7.5, seed=42, h=512, wd=512):
    emb = encode([negative, prompt])                             # 1. nhánh không điều kiện + có điều kiện
    g = torch.Generator(dev).manual_seed(seed)                   # 2. nhiễu ban đầu từ seed
    z = torch.randn(1, 4, h // 8, wd // 8, generator=g, device=dev, dtype=torch.float16)
    sched.set_timesteps(steps)                                   # 3. chọn các bước (999, 966, 932, ...)
    z = z * sched.init_noise_sigma
    for t in sched.timesteps:                                    # 4. lặp
        inp = sched.scale_model_input(torch.cat([z, z]), t)
        eps_u, eps_c = unet(inp, t, encoder_hidden_states=emb).sample.chunk(2)
        eps = eps_u + w * (eps_c - eps_u)                        #    CFG
        z = sched.step(eps, t, z).prev_sample                    #    scheduler cập nhật latent
    img = vae.decode(z / vae.config.scaling_factor).sample       # 5. giải mã
    img = ((img.clamp(-1, 1) + 1) / 2)[0].permute(1, 2, 0).float().cpu().numpy()
    return (img * 255).round().astype("uint8")                   # (512, 512, 3)

In [ ]:
import torch
from diffusers import UNet2DConditionModel, AutoencoderKL, DPMSolverMultistepScheduler
torch.manual_seed(0)
unet = UNet2DConditionModel(sample_size=16, block_out_channels=(32, 64), layers_per_block=1, cross_attention_dim=32,
                            down_block_types=("CrossAttnDownBlock2D", "DownBlock2D"),
                            up_block_types=("UpBlock2D", "CrossAttnUpBlock2D"), attention_head_dim=8)
vae = AutoencoderKL(block_out_channels=(32, 64), down_block_types=("DownEncoderBlock2D",) * 2,
                    up_block_types=("UpDecoderBlock2D",) * 2, latent_channels=4, norm_num_groups=16)
sched = DPMSolverMultistepScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear")
emb = torch.randn(2, 77, 32)                                   # [nhánh không điều kiện, nhánh có điều kiện]

z = torch.randn(1, 4, 16, 16) * sched.init_noise_sigma
sched.set_timesteps(30)
print("các bước:", sched.timesteps[:4].tolist(), "...")
with torch.no_grad():
    for t in sched.timesteps:
        inp = sched.scale_model_input(torch.cat([z, z]), t)
        eps_u, eps_c = unet(inp, t, encoder_hidden_states=emb).sample.chunk(2)
        z = sched.step(eps_u + 7.5 * (eps_c - eps_u), t, z).prev_sample
    img = vae.decode(z / 0.18215).sample
print("latent", tuple(z.shape), "→ ảnh", tuple(img.shape))     # VAE nhỏ này giảm 2 lần mỗi cạnh; VAE của SD giảm 8 lần

các bước: [999, 966, 932, 899] ...
latent (1, 4, 16, 16) → ảnh (1, 3, 32, 32)


## Bài tập 5: Đọc pipeline {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần mạng: tải Stable Diffusion 1.5, chạy trên Colab
import torch
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16)
count = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print(f"U-Net {count(pipe.unet):.0f}M | text encoder {count(pipe.text_encoder):.0f}M | VAE {count(pipe.vae):.0f}M")
print("scaling factor:", pipe.vae.config.scaling_factor)
print("scheduler:", type(pipe.scheduler).__name__, "| max tokens:", pipe.tokenizer.model_max_length)
print("cross-attention dim:", pipe.unet.config.cross_attention_dim)